# Geographic Mapping with China Data

This notebook shows how to merge a province-level panel dataset with a shapefile and draw a basic choropleth map.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
MODULE_OUTPUT_DIR = OUTPUT_DIR / "07_geographic_mapping"
MODULE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODULE_OUTPUT_DIR

**Optional setup**

In [ ]:
%pip install -q openpyxl geopandas

In [ ]:
import pandas as pd

# Read the panel data and keep one year for mapping.
data1 = pd.read_excel(
    DATA_DIR / "china" / "china_panel_data.xlsx",
    sheet_name=0,
).rename(columns={"地区": "pr_name"}).query("年份 == 2013")

data1.head()

In [ ]:
import geopandas as gpd

# Read the province shapefile.
gdf = gpd.read_file(DATA_DIR / "maps" / "china_provinces.shp")

# Remove suffixes so that province names match the panel data.
gdf["pr_name"] = gdf["pr_name"].str.rstrip("市省自治区壮族回族维吾尔特别行政区")

gdf.plot()

In [ ]:
gdf

In [ ]:
gdf = gdf.merge(data1, on="pr_name", how="outer")

In [ ]:
gdf

In [ ]:
# Create a short English alias for the mapped variable.
gdf["population_10k_persons"] = gdf["总人口（万人）"]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))

gdf.plot(
    ax=ax,
    column="population_10k_persons",
    legend=True,
    cmap="winter",
    missing_kwds={"color": "grey", "hatch": "///"},
    legend_kwds={
        "shrink": 0.7,
        "ticks": [3000, 4000, 6000, 8000, 10000],
        "label": "Population (10,000 persons)",
    },
)

ax.title.set_text("Population Distribution by Province, 2013")

plt.savefig(MODULE_OUTPUT_DIR / "population_distribution_2013.png", bbox_inches="tight")
plt.show()